In [4]:
df_sales = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load("Files/bronze/SalesOrderHeader.csv")

df_sales = df_sales.toDF(
    "SalesOrderID", "RevisionNumber", "OrderDate", "DueDate", "ShipDate",
    "Status", "OnlineOrderFlag", "SalesOrderNumber", "PurchaseOrderNumber",
    "AccountNumber", "CustomerID", "SalesPersonID", "TerritoryID",
    "BillToAddressID", "ShipToAddressID", "ShipMethodID", "CreditCardID",
    "CreditCardApprovalCode", "CurrencyRateID", "SubTotal", "TaxAmt",
    "Freight", "TotalDue", "Comment", "rowguid", "ModifiedDate"
)

display(df_sales.limit(5))

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8599713e-cdd6-4572-b29a-7440dbafe267)

In [7]:
df_product = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load("Files/bronze/Product.csv")

df_product = df_product.toDF(
    "ProductID", "Name", "ProductNumber", "MakeFlag", "FinishedGoodsFlag",
    "Color", "SafetyStockLevel", "ReorderPoint", "StandardCost", "ListPrice",
    "Size", "SizeUnitMeasureCode", "WeightUnitMeasureCode", "Weight",
    "DaysToManufacture", "ProductLine", "Class", "Style", "ProductSubcategoryID",
    "ProductModelID", "SellStartDate", "SellEndDate", "DiscontinuedDate",
    "rowguid", "ModifiedDate"
)

display(df_product.limit(5))
df_product.write.format("delta").mode("overwrite").saveAsTable("bronze_product")

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb7c84cc-6303-4650-851a-c1829c93ee4b)

In [8]:
df_territory = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load("Files/bronze/SalesTerritory.csv")

df_territory = df_territory.toDF(
    "TerritoryID", "Name", "CountryRegionCode", "Group", "SalesYTD",
    "SalesLastYear", "CostYTD", "CostLastYear", "rowguid", "ModifiedDate"
)

display(df_territory.limit(5))
df_territory.write.format("delta").mode("overwrite").saveAsTable("bronze_sales_territory")

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a54eb662-b2f8-45fa-a6c8-599436ac0b1c)

In [9]:
df_detail = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load("Files/bronze/SalesOrderDetail.csv")

df_detail = df_detail.toDF(
    "SalesOrderID", "SalesOrderDetailID", "CarrierTrackingNumber",
    "OrderQty", "ProductID", "SpecialOfferID", "UnitPrice",
    "UnitPriceDiscount", "LineTotal", "rowguid", "ModifiedDate"
)

display(df_detail.limit(5))
df_detail.write.format("delta").mode("overwrite").saveAsTable("bronze_sales_order_detail")

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b517f8bb-3775-42fe-9d88-dfa0736b27fa)

In [10]:
df_customer = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .load("Files/bronze/Customer.csv")

df_customer = df_customer.toDF(
    "CustomerID", "PersonID", "StoreID", "TerritoryID",
    "AccountNumber", "rowguid", "ModifiedDate"
)

display(df_customer.limit(5))
df_customer.write.format("delta").mode("overwrite").saveAsTable("bronze_customer")

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b689f93-e5a4-488d-b186-3929254feb0a)

In [11]:
from pyspark.sql.functions import col

# 清洗 SalesOrderHeader
df_header_clean = spark.table("bronze_sales_order_header") \
    .dropDuplicates(["SalesOrderID"]) \
    .filter(col("OrderDate").isNotNull())

# 清洗 SalesOrderDetail
df_detail_clean = spark.table("bronze_sales_order_detail") \
    .dropDuplicates(["SalesOrderDetailID"]) \
    .filter(col("ProductID").isNotNull())

# 清洗 Product
df_product_clean = spark.table("bronze_product") \
    .dropDuplicates(["ProductID"])

# 清洗 SalesTerritory
df_territory_clean = spark.table("bronze_sales_territory") \
    .dropDuplicates(["TerritoryID"])

# 清洗 Customer
df_customer_clean = spark.table("bronze_customer") \
    .dropDuplicates(["CustomerID"])

print("Header:", df_header_clean.count())
print("Detail:", df_detail_clean.count())
print("Product:", df_product_clean.count())
print("Territory:", df_territory_clean.count())
print("Customer:", df_customer_clean.count())

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 13, Finished, Available, Finished, False)

Header: 31465
Detail: 121317
Product: 504
Territory: 10
Customer: 19820


In [12]:
from pyspark.sql.functions import monotonically_increasing_id

# DimProduct
df_dim_product = df_product_clean.select(
    "ProductID", "Name", "ProductNumber", "Color",
    "StandardCost", "ListPrice", "ProductLine", "Class"
).withColumn("ProductKey", monotonically_increasing_id())

# DimTerritory
df_dim_territory = df_territory_clean.select(
    "TerritoryID", "Name", "CountryRegionCode", "Group"
).withColumn("TerritoryKey", monotonically_increasing_id())

# DimCustomer
df_dim_customer = df_customer_clean.select(
    "CustomerID", "PersonID", "StoreID", "TerritoryID", "AccountNumber"
).withColumn("CustomerKey", monotonically_increasing_id())

# 保存
df_dim_product.write.format("delta").mode("overwrite").saveAsTable("gold_dim_product")
df_dim_territory.write.format("delta").mode("overwrite").saveAsTable("gold_dim_territory")
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold_dim_customer")

display(df_dim_product.limit(5))

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5bae707d-c509-45b4-9541-a4efdd286f4e)

In [13]:
from pyspark.sql.functions import col

# 第一步:Detail 关联 Header,拿到订单级别信息(日期、客户ID)
df_fact_base = df_detail_clean.join(
    df_header_clean.select("SalesOrderID", "OrderDate", "CustomerID", "Status"),
    on="SalesOrderID",
    how="left"
)

# 第二步:关联 Customer,拿到 TerritoryID(客户所在地区)
df_fact_base = df_fact_base.join(
    df_customer_clean.select("CustomerID", "TerritoryID"),
    on="CustomerID",
    how="left"
)

# 第三步:把自然键换成维度表的代理键(Key)
df_fact_sales = df_fact_base \
    .join(df_dim_product.select("ProductID", "ProductKey"), on="ProductID", how="left") \
    .join(df_dim_territory.select("TerritoryID", "TerritoryKey"), on="TerritoryID", how="left") \
    .join(df_dim_customer.select("CustomerID", "CustomerKey"), on="CustomerID", how="left")

# 第四步:只保留事实表该有的字段——代理键 + 日期 + 度量值
df_fact_sales = df_fact_sales.select(
    "SalesOrderID",
    "SalesOrderDetailID",
    "ProductKey",
    "TerritoryKey",
    "CustomerKey",
    "OrderDate",
    "OrderQty",
    "UnitPrice",
    "UnitPriceDiscount",
    "LineTotal"
)

# 保存
df_fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold_fact_sales")

print("行数:", df_fact_sales.count())
display(df_fact_sales.limit(10))

StatementMeta(, 3da84103-a453-4b42-9252-48a51c28d001, 15, Finished, Available, Finished, False)

行数: 121317


SynapseWidget(Synapse.DataFrame, e73f689e-5fa2-479a-ad5a-0f7b74b9e7d0)